# 04 - Feature Engineering

Turns the joined table into the exact feature set the models will see.
The rule enforced throughout `src/features.py`: a feature may only use
information that would genuinely have been **observable at the moment
the order was placed**. The historical delay-rate features are where
that rule is easiest to break, so they get the most care below.

In [1]:
import sys
sys.path.append("../src")
import pandas as pd
from features import build_feature_table, FEATURE_COLUMNS, TARGET_COLUMN

df = pd.read_csv("../data/processed/ontime_joined_raw.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
])

feat = build_feature_table(df)
print("Feature table:", feat.shape)
feat[FEATURE_COLUMNS].dtypes

Feature table: (96470, 53)


n_items                                   int64
n_distinct_products                       int64
n_distinct_sellers                        int64
total_price                             float64
avg_item_price                          float64
max_item_price                          float64
total_freight                           float64
avg_freight                             float64
total_order_value                       float64
product_category                            str
total_weight_g                          float64
avg_weight_g                            float64
total_volume_cm3                        float64
avg_photos_qty                          float64
avg_name_length                         float64
avg_description_length                  float64
purchase_dow                              int32
purchase_month                            int32
purchase_hour                             int32
is_weekend_purchase                       int64
approval_delay_hours                    

## What "historical delay rate" is allowed to mean

A naive `groupby(seller).is_delayed.mean()` averages a seller's rate over
ALL of their orders - including the order being predicted and orders
that happen later. That is plain leakage.

The first version of this project fixed that by averaging only orders
*purchased* earlier. That still leaks, more subtly: an order bought three
days ago has **not been delivered yet**, so its `is_delayed` label does
not exist at prediction time. Delivery takes ~10 days at the median, so
the most recent - and, because delays cluster in time, the most
informative - part of that "history" was not actually knowable.

`add_historical_rates()` now defines the history of an order placed at
time *t* as the same-group orders **delivered before *t***. An order can
never be in its own history, and nothing from the future is used - not
even in the fallback for a group with no history yet, which is the
platform-wide rate *as known at time t* rather than an all-time average.
(Notebook 07 measures what this change did to model performance.)

In [2]:
example_seller = feat["primary_seller_id"].value_counts().index[0]
cols = ["order_purchase_timestamp", "order_delivered_customer_date", "is_delayed", "seller_historical_delay_rate"]
feat[feat["primary_seller_id"] == example_seller].sort_values("order_purchase_timestamp")[cols].head(8)

,order_purchase_timestamp,order_delivered_customer_date,is_delayed,seller_historical_delay_rate
69159,2017-02-17 07:39:19,2017-02-23 13:05:03,0,0.002892
6424,2017-02-23 10:36:35,2017-03-02 14:18:07,0,0.002295
95905,2017-02-24 20:28:53,2017-03-08 11:38:02,0,0.000000
94214,2017-02-27 09:27:13,2017-03-10 07:58:50,0,0.000000
25155,2017-02-28 20:57:21,2017-03-03 07:05:10,0,0.000000
94747,2017-03-01 20:28:48,2017-03-08 11:47:21,0,0.000000
2518,2017-03-03 14:24:08,2017-03-13 14:38:04,0,0.000000
14530,2017-03-03 18:29:14,2017-03-17 14:45:49,0,0.000000


Read down the last column against the two date columns: the rate for an
order only changes after an earlier order of that seller has been
*delivered*, not merely purchased. Orders with no delivered history for
that seller yet use the platform-wide rate known at that moment.

### Independent verification on real data

Rather than trust the implementation, recompute a sample of orders'
seller history the slow, obvious way - filter the raw table to that
seller's orders delivered before this order's purchase time and average
them - and confirm it matches the feature exactly.

In [3]:
import numpy as np
sample = feat.sample(400, random_state=0)
checked = mismatched = 0
for _, r in sample.iterrows():
    hist = feat[(feat["primary_seller_id"] == r["primary_seller_id"])
                & (feat["order_delivered_customer_date"] < r["order_purchase_timestamp"])]
    if len(hist) == 0:
        continue                      # fallback case - covered separately
    checked += 1
    if abs(hist["is_delayed"].mean() - r["seller_historical_delay_rate"]) > 1e-9:
        mismatched += 1
print(f"Orders with delivered seller history checked: {checked}   mismatches: {mismatched}")
assert mismatched == 0

# and no order may see its own outcome: its delivery is always after its purchase
assert (feat["order_delivered_customer_date"] >= feat["order_purchase_timestamp"]).all()
print("Every order is delivered on/after its own purchase, so it can never be in its own history.")

Orders with delivered seller history checked: 378   mismatches: 0
Every order is delivered on/after its own purchase, so it can never be in its own history.


## Final feature list

In [4]:
for c in FEATURE_COLUMNS:
    print("-", c)

- n_items
- n_distinct_products
- n_distinct_sellers
- total_price
- avg_item_price
- max_item_price
- total_freight
- avg_freight
- total_order_value
- product_category
- total_weight_g
- avg_weight_g
- total_volume_cm3
- avg_photos_qty
- avg_name_length
- avg_description_length
- purchase_dow
- purchase_month
- purchase_hour
- is_weekend_purchase
- approval_delay_hours
- promised_delivery_days
- customer_state
- seller_state
- same_state
- seller_customer_distance_km
- payment_type
- n_payment_installments
- total_payment_value
- n_payment_methods
- seller_historical_delay_rate
- category_historical_delay_rate
- customer_state_historical_delay_rate


In [5]:
keep = ["order_id", TARGET_COLUMN, "order_purchase_timestamp"] + FEATURE_COLUMNS
feat[keep].to_csv("../data/processed/ontime_model_ready.csv", index=False)
print("Saved -> data/processed/ontime_model_ready.csv", feat[keep].shape)

Saved -> data/processed/ontime_model_ready.csv (96470, 36)


## Sanity check: does any feature perfectly separate the classes?
(a suspiciously perfect correlation would be a red flag for hidden leakage)

In [6]:
numeric_feats = feat[FEATURE_COLUMNS].select_dtypes("number").columns
feat[list(numeric_feats) + [TARGET_COLUMN]].corr()[TARGET_COLUMN].sort_values(ascending=False)

is_delayed                              1.000000
customer_state_historical_delay_rate    0.082136
seller_customer_distance_km             0.068282
avg_freight                             0.042237
seller_historical_delay_rate            0.032626
approval_delay_hours                    0.029487
total_freight                           0.025328
avg_weight_g                            0.023756
avg_item_price                          0.021348
max_item_price                          0.020648
total_order_value                       0.017496
total_payment_value                     0.017477
total_weight_g                          0.016530
total_price                             0.015700
category_historical_delay_rate          0.014244
total_volume_cm3                        0.011428
n_payment_installments                  0.009003
purchase_hour                           0.008667
avg_name_length                         0.005449
avg_description_length                  0.000724
n_payment_methods   